In [1]:
import torch

print('Có GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Tên GPU:', torch.cuda.get_device_name(0))

Có GPU: True
Tên GPU: Tesla T4


In [5]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)
# Khi hỏi cấp quyền, chọn email tài khoản hiện tại.

root = Path('/content/drive/MyDrive/CS313_Nhom5')
print('Có data:', (root / 'data').is_dir())
print('Có E3a:', (root / 'hoang_khang/E3a').is_dir())

Mounted at /content/drive
Có data: True
Có E3a: True


In [16]:
from google.colab import files
uploaded = files.upload()

Saving DeepSVDD_9_RUNS_20261126T164346Z.zip to DeepSVDD_9_RUNS_20261126T164346Z (1).zip


In [9]:
import ast
import hashlib
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, Subset

# Kiểm tra đủ 3 file đã upload
for name in [
    'NHK_code (11).ipynb',
    'usad_colab_cells.py',
    'deepsvdd_upstream_colab.py',
]:
    assert Path('/content', name).is_file(), f'Thiếu file: {name}'

# Chỉ lấy checksum và lớp Dataset; KHÔNG chạy notebook cũ
nb = json.loads(
    Path('/content/NHK_code (11).ipynb').read_text(
        encoding='utf-8-sig'
    )
)

constants, definitions = {}, {}

for cell in nb['cells']:
    if cell.get('cell_type') != 'code':
        continue
    try:
        tree = ast.parse(''.join(cell.get('source', [])))
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in {
                    'PSM_EXPECTED_SHA', 'EXTRA_DATA_REFERENCES'
                }:
                    constants[target.id] = ast.literal_eval(node.value)

        if isinstance(node, ast.ClassDef) and node.name == 'PSMWindowDataset':
            definitions[node.name] = node

assert set(constants) == {'PSM_EXPECTED_SHA', 'EXTRA_DATA_REFERENCES'}
assert 'PSMWindowDataset' in definitions
globals().update(constants)

exec(compile(
    ast.Module(
        body=[definitions['PSMWindowDataset']],
        type_ignores=[]
    ),
    'dataset_only', 'exec'
), globals())

# Chỉ lấy helper đọc dữ liệu chung; KHÔNG chạy USAD
tree = ast.parse(
    Path('/content/usad_colab_cells.py').read_text(encoding='utf-8-sig')
)
helpers = [
    n for n in tree.body
    if isinstance(n, ast.FunctionDef)
    and n.name == 'load_usad_shared_data'
]
assert len(helpers) == 1

exec(compile(
    ast.Module(body=helpers, type_ignores=[]),
    'shared_loader_only', 'exec'
), globals())

# Nạp runner SVDD mới, dùng output riêng
exec(compile(
    Path('/content/deepsvdd_upstream_colab.py').read_text(
        encoding='utf-8-sig'
    ),
    'deepsvdd_upstream_colab.py', 'exec'
), globals())

setup_svdd_upstream(
    output_root='/content/drive/MyDrive/NHK_DeepSVDD_UPSTREAM/E3a'
)

Đã nạp source và kiểm tra quyền ghi. CHƯA train.
Output riêng: /content/drive/MyDrive/NHK_DeepSVDD_UPSTREAM/E3a


In [10]:
PSM16_DIR = run_svdd_upstream_if_missing('PSM', 16)

DeepSVDD upstream_eval_bn PSM seed16: train/val/test=10595/2648/8779
Epoch1=train, epoch2..20=eval nhưng vẫn cập nhật gradient. Output: /content/drive/MyDrive/NHK_DeepSVDD_UPSTREAM/E3a/DeepSVDD_upstream_eval_bn_PSM_seed16_20261126T163306_125411Z
PSM seed16 upstream_eval_bn epoch 01/20 | loss=17.058444 AUROC=0.693652 AP=0.512509
PSM seed16 upstream_eval_bn epoch 02/20 | loss=2.024969 AUROC=0.705915 AP=0.519514
PSM seed16 upstream_eval_bn epoch 03/20 | loss=1.043222 AUROC=0.695680 AP=0.504444
PSM seed16 upstream_eval_bn epoch 04/20 | loss=0.688031 AUROC=0.690367 AP=0.499034
PSM seed16 upstream_eval_bn epoch 05/20 | loss=0.560846 AUROC=0.688779 AP=0.499875
PSM seed16 upstream_eval_bn epoch 06/20 | loss=0.502914 AUROC=0.689705 AP=0.504728
PSM seed16 upstream_eval_bn epoch 07/20 | loss=0.470017 AUROC=0.691459 AP=0.510355
PSM seed16 upstream_eval_bn epoch 08/20 | loss=0.445884 AUROC=0.694062 AP=0.516854
PSM seed16 upstream_eval_bn epoch 09/20 | loss=0.426302 AUROC=0.696677 AP=0.521822
PSM se

In [11]:
assert find_completed_upstream('PSM', 16) is not None

for dataset in ['PSM', 'MSL', 'SMD']:
    for seed in [16, 17, 18]:
        run_svdd_upstream_if_missing(dataset, seed)

SUMMARY_DIR = summarize_svdd_upstream()

BỎ QUA — đã hoàn tất upstream_eval_bn PSM seed16
DeepSVDD upstream_eval_bn PSM seed17: train/val/test=10595/2648/8779
Epoch1=train, epoch2..20=eval nhưng vẫn cập nhật gradient. Output: /content/drive/MyDrive/NHK_DeepSVDD_UPSTREAM/E3a/DeepSVDD_upstream_eval_bn_PSM_seed17_20261126T163346_185990Z
PSM seed17 upstream_eval_bn epoch 01/20 | loss=17.546094 AUROC=0.578817 AP=0.450751
PSM seed17 upstream_eval_bn epoch 02/20 | loss=2.370322 AUROC=0.616478 AP=0.484974
PSM seed17 upstream_eval_bn epoch 03/20 | loss=1.121954 AUROC=0.659107 AP=0.519361
PSM seed17 upstream_eval_bn epoch 04/20 | loss=0.716709 AUROC=0.670965 AP=0.529176
PSM seed17 upstream_eval_bn epoch 05/20 | loss=0.583943 AUROC=0.672408 AP=0.530933
PSM seed17 upstream_eval_bn epoch 06/20 | loss=0.529658 AUROC=0.672113 AP=0.531226
PSM seed17 upstream_eval_bn epoch 07/20 | loss=0.496655 AUROC=0.671878 AP=0.531432
PSM seed17 upstream_eval_bn epoch 08/20 | loss=0.472897 AUROC=0.669817 AP=0.530773
PSM seed17 upstream_eval_bn epoch 09/20 

In [12]:
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

root = Path('/content/drive/MyDrive/NHK_DeepSVDD_UPSTREAM/E3a')
assert root.is_dir(), 'Chưa thấy thư mục kết quả; kiểm tra mount Drive.'

wanted = [
    'metrics.csv', 'epoch_metrics.csv', 'epoch_history.csv',
    'run_config.json', 'config.json', 'completed.json',
    'adapter_notes.json', 'data_manifest.json', 'train.log',
    'validation_scores.csv', 'test_scores.csv',
]

# Dùng inventory của bản tổng hợp mới nhất để chọn đúng 9 lượt
summaries = sorted(root.glob('SUMMARY_upstream_eval_bn_*'))
assert summaries, 'Chưa thấy bản tổng hợp 9 lượt.'

import pandas as pd
inventory = pd.read_csv(summaries[-1] / 'run_inventory.csv')
assert len(inventory) == 9
assert not inventory.duplicated(['dataset', 'seed']).any()

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
zip_path = Path(f'/content/DeepSVDD_9_RUNS_{stamp}.zip')

with ZipFile(zip_path, 'w', ZIP_DEFLATED) as z:
    for run_dir in inventory['run_dir']:
        folder = Path(run_dir)
        for name in wanted:
            p = folder / name
            assert p.is_file(), f'Thiếu file: {p}'
            z.write(p, arcname=f'{folder.name}/{name}')

    for p in summaries[-1].glob('*.csv'):
        z.write(p, arcname=f'{summaries[-1].name}/{p.name}')

print('Đã tạo:', zip_path.name)
files.download(str(zip_path))

Đã tạo: DeepSVDD_9_RUNS_20261126T164346Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
from pathlib import Path

print('TK1:', Path('/content/drive/MyDrive/CS313_Nhom5/hoang_khang/E3a').is_dir())
print('TK2:', Path('/content/drive/MyDrive/NHK_GANF_TAI_KHOAN_2/E3a').is_dir())

TK1: True
TK2: True


In [17]:
from pathlib import Path

script = Path('/content/merge_nhk_svdd_p99_colab.py')
exec(compile(script.read_text(encoding='utf-8-sig'), str(script), 'exec'), globals())

OUT45, RESULTS45, SUMMARY45 = merge_nhk_p99(
    old_results_path='/content/results_45_runs.csv',
    old_inventory_path='/content/run_inventory.csv',
    svdd_zip_path='/content/DeepSVDD_9_RUNS_20261126T164346Z.zip',
)

display(SUMMARY45)

Đã nạp script ghép/P99. CHƯA tính và KHÔNG train; gọi merge_nhk_p99(...) để bắt đầu.
OK MTGFlow  PSM seed16: score/label/metric hợp lệ; đã tính P99.
OK MTGFlow  PSM seed17: score/label/metric hợp lệ; đã tính P99.
OK MTGFlow  PSM seed18: score/label/metric hợp lệ; đã tính P99.
OK MTGFlow  MSL seed16: score/label/metric hợp lệ; đã tính P99.
OK MTGFlow  MSL seed17: score/label/metric hợp lệ; đã tính P99.
OK MTGFlow  MSL seed18: score/label/metric hợp lệ; đã tính P99.
OK MTGFlow  SMD seed16: score/label/metric hợp lệ; đã tính P99.
OK MTGFlow  SMD seed17: score/label/metric hợp lệ; đã tính P99.
OK MTGFlow  SMD seed18: score/label/metric hợp lệ; đã tính P99.
OK DeepSVDD PSM seed16: score/label/metric hợp lệ; đã tính P99.
OK DeepSVDD PSM seed17: score/label/metric hợp lệ; đã tính P99.
OK DeepSVDD PSM seed18: score/label/metric hợp lệ; đã tính P99.
OK DeepSVDD MSL seed16: score/label/metric hợp lệ; đã tính P99.
OK DeepSVDD MSL seed17: score/label/metric hợp lệ; đã tính P99.
OK DeepSVDD MSL see

,model,variant,dataset,n_seeds,auroc_mean,auroc_std,auprc_mean,auprc_std,f1_mean,f1_std,flagged_rate_mean,flagged_rate_std,precision_mean,precision_std,recall_mean,recall_std,runtime_s_mean,runtime_s_std
0,ALOCC,mtgflow_cnn_width16_safe_probe,MSL,3,0.507465,0.044002,0.170076,0.027530,0.074756,0.017396,0.034026,0.032650,0.263308,0.121524,0.047860,0.018817,14.088809,0.209767
1,ALOCC,mtgflow_cnn_width16_safe_probe,PSM,3,0.492288,0.072702,0.369786,0.085663,0.055047,0.074480,0.010100,0.012936,0.823843,0.132321,0.029426,0.040418,26.207391,0.430930
2,ALOCC,mtgflow_cnn_width16_safe_probe,SMD,3,0.509665,0.040259,0.082696,0.001843,0.044482,0.042900,0.026606,0.021324,0.098226,0.047140,0.034667,0.037565,129.455272,0.766296
3,DeepSVDD,upstream_eval_bn,MSL,3,0.469198,0.026780,0.192350,0.007359,0.151422,0.031545,0.039500,0.018397,0.355020,0.057198,0.099427,0.031334,11.218927,0.115091
4,DeepSVDD,upstream_eval_bn,PSM,3,0.674872,0.055947,0.529266,0.084132,0.356286,0.045570,0.161256,0.121583,0.630497,0.188550,0.275084,0.107502,14.397034,1.179275
5,DeepSVDD,upstream_eval_bn,SMD,3,0.744722,0.021547,0.224940,0.020172,0.191091,0.048991,0.023716,0.004752,0.370602,0.044795,0.129573,0.038451,91.623317,0.804985
6,GANF,official_water_augLag_shared_standard,MSL,3,0.596790,0.011448,0.225927,0.004497,0.143595,0.025585,0.062984,0.053299,0.282612,0.090927,0.108864,0.050215,87.633752,2.547936
7,GANF,official_water_augLag_shared_standard,PSM,3,0.689329,0.032072,0.541360,0.047307,0.189972,0.157940,0.049702,0.048063,0.846513,0.155385,0.117824,0.103982,87.265550,2.107583
8,GANF,official_water_augLag_shared_standard,SMD,3,0.660570,0.016393,0.172080,0.023233,0.096848,0.056470,0.013308,0.010990,0.337673,0.060978,0.060786,0.043913,661.919831,34.951633
9,MTGFlow,base,MSL,3,0.681755,0.032062,0.276125,0.026808,0.188595,0.087551,0.057373,0.036409,0.341120,0.072084,0.142231,0.093507,81.193214,1.026504
